In [ ]:
# --- Environment & imports ---
# NOTE: GPU calculator must be imported before other md imports.
%load_ext autoreload
%autoreload 2

import os
import sys
# Make local package importable.
sys.path.append("/root/limlab01/kaistai/25DFT/QHFlow/src")

import md.scflow_calculator_gpu  # GPU backend (import early)

# Threading and IO controls (tune as needed).
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # pin to a single GPU if needed


from dft_process.dft_process_utils import *

In [ ]:
# --- Dataset selection ---
MD17_DATASETS = {
    0: "water",
    1: "ethanol",
    2: "malondialdehyde",
    3: "uracil",
    4: "rmd-ethanol",
    5: "rmd-salicylic_acid",
    6: "rmd-naphthalene",
    7: "rmd-aspirin",
}

DATA_INDEX = 1  # choose dataset here
DATASET_NAME = MD17_DATASETS[DATA_INDEX]

# Load dataset wrapper and its model path config.
md17_experiment = MD17Experiment(data_index=DATA_INDEX)
md17_experiment.set_model_path()

# --- Checkpoint selection (per dataset index) ---
# Uncomment paths that exist on your machine.
CUR_CKPT_BY_INDEX = {
    # 0: "/root/25DFT/QHFlow/src/outputs/water/QHFlow_so2_v5_1_middle_b10-water/checkpoints/last.ckpt",
    1: "/root/25DFT/QHFlow/src/outputs/ethanol/QHFlow_so2_v5_1_middle_b10-ethanol/checkpoints/last.ckpt",
    2: "/root/25DFT/QHFlow/src/outputs/malondialdehyde/QHFlow_so2_v5_1_middle_b10-malondialdehyde/checkpoints/last.ckpt",
    3: "/root/25DFT/QHFlow/src/outputs/uracil/QHFlow_so2_v5_1_middle_b10-uracil/checkpoints/last.ckpt",
    # 4: "/root/25DFT/QHFlow/src/outputs/rmd-ethanol/QHFlow_so2_v5_1_middle_b10-rmd-ethanol/checkpoints/last.ckpt",
    5: "/root/25DFT/QHFlow/src/outputs/rmd-salicylic_acid/QHFlow_so2_v5_1_middle_b10-rmd-salicylic_acid/checkpoints/last.ckpt",
    6: "/root/25DFT/QHFlow/src/outputs/rmd-naphthalene/QHFlow_so2_v5_1_middle_b10-rmd-naphthalene/checkpoints/last.ckpt",
    7: "/root/25DFT/QHFlow/src/outputs/rmd-aspirin/QHFlow_so2_v5_1_middle_b10-rmd-aspirin/checkpoints/last.ckpt",
}

if DATA_INDEX not in CUR_CKPT_BY_INDEX:
    raise ValueError(f"No checkpoint registered for DATA_INDEX={DATA_INDEX} ({DATASET_NAME}).")

cur_ckpt = CUR_CKPT_BY_INDEX[DATA_INDEX]

In [ ]:
# --- Optional: manual geometry example (ethanol) ---
# This is a small helper for quick sanity checks without the dataset.
sample_ethanol = [
    (6,  [-0.28630036, -0.47787276,  0.03603168]),  # C
    (6,  [-0.92012881,  0.96245698,  0.23729976]),  # C
    (8,  [ 1.10116376, -0.40535771, -0.20377836]),  # O
    (1,  [-0.49412140, -1.15617614,  0.94202625]),  # H
    (1,  [-0.80417700, -0.91072034, -0.80315132]),  # H
    (1,  [-1.91334152,  0.70271240,  0.63912332]),  # H
    (1,  [-0.36050056,  1.60555901,  0.92692388]),  # H
    (1,  [-0.93001164,  1.37368891, -0.73671183]),  # H
    (1,  [ 1.39790227, -0.95652028, -0.99060288]),  # H
]

sample_ethanol_atoms = [a[0] for a in sample_ethanol]
sample_ethanol_coords = [a[1] for a in sample_ethanol]
sample_ethanol_atoms, sample_ethanol_coords

In [ ]:
# --- Build SCFlow calculator (GPU) ---
sccalculator = md.scflow_calculator_gpu.SCFlowRKSCalculator()

In [ ]:
# --- Configure calculator options ---
# Select config based on dataset group.
SCCAL_CONFIGS = {
    "md17": {
        "basis": "def2-SVP",
        "functional": "pbe",
        "units": "ang",
        "model_length_unit": "bohr",
        "mf_init_functional": "pbe, pbe",
        "init_density_fit": False,
    },
    "rmd17": {
        "basis": "def2-SVP",
        "functional": "pbe",
        "units": "ang",
        "model_length_unit": "ang",
        "mf_init_functional": "pbe",
        "density_fit": True,
        "init_density_fit": True,
    },
    "qh9": {
        "basis": "def2-SVP",
        "functional": "b3lyp",
        "units": "ang",
        "model_length_unit": "ang",
        "mf_init_functional": "b3lyp",
    },
}

if DATA_INDEX in [0, 1, 2, 3]:
    dataset_group = "md17"
elif DATA_INDEX in [4, 5, 6, 7]:
    dataset_group = "rmd17"
else:
    raise ValueError(f"Unsupported DATA_INDEX={DATA_INDEX} for SCFlow config.")

sccal_config = SCCAL_CONFIGS[dataset_group]

# Load the checkpoint into the calculator.
sccalculator.set_model(cur_ckpt, **sccal_config)

In [ ]:
# --- Load one sample from the dataset ---
sample_idx = 1  # change to any valid index
sample = md17_experiment.dataset[sample_idx]

# Inspect reference energy from the dataset.
sample.energy

In [ ]:
# Convert QHData sample to ASE atoms (input is in bohr).
ase_atoms = md.scflow_calculator_gpu.QHData_to_atoms(sample, unit="bohr")

In [ ]:
# Quick sanity check on the ASE object.
ase_atoms

In [ ]:
# Run SCFlow inference.
# units: output units; model_length_unit: length unit used during training.
properties = ["energy", "forces"]
system_changes = ["positions"]

sccalculator.calculate(
    ase_atoms,
    properties,
    system_changes,
    units="ang",
)

In [ ]:
# Example: check overlap matrix shape.
sccalculator.ovlp.shape

In [ ]:
# Optional: compare with a plain RKS calculation.
rkscalculator = md.scflow_calculator.RKSCalculator()
rkscalculator.calculate(ase_atoms, properties, system_changes, units="ang")

rkscalculator.results["energy"], rkscalculator.results["forces"]

# Frontier orbitals (if needed).
homo = rkscalculator.homo
lumo = rkscalculator.lumo